# SK KB Evaluations

# Introduction

This notebook analyzes the baseline Slovak knowledge-base evaluation defined in `configs/skkb/skkb_exp_001_baseline.yaml` together with the corresponding checkpoint results in `notebooks/skkb/checkpoints/evals_skkb_exp_001_baseline_medium.csv`. The goal is diagnostic: not just to assign an overall pass/fail score, but to localize where an SKKB case breaks across the end-to-end pipeline.

Compared with earlier SKKB checks that emphasized deterministic retrieval metrics and coarse answer grading, this evaluation adds an LLM-as-a-judge rubric plus structured failure annotations. That lets us separate query-quality issues from scope/routing problems, retriever and reranker misses, selected-context insufficiency, KB content gaps, answer-generation defects, grounding failures, and Slovak language compliance issues.

Evaluation dimensions:

| Dimension | What it measures | Why it matters |
| --- | --- | --- |
| `query_clarity` | Whether the user query is clear, ambiguous, or unclear. | Prevents over-attributing failures to retrieval or generation when the input itself is weak. |
| `case_scope_fit` | Whether the case is clearly KB, API, KB and API, ambiguous, or out-of-scope. | Separates true KB defects from routing and scope problems. |
| `selection_semantic_relevance` | Whether the selected ENUMs are semantically the right ones. | Diagnoses reranker and selection quality. |
| `selected_context_sufficiency` | Whether the selected KB context is sufficient to answer the case. | Tells us if the bot was given enough evidence. |
| `optimal_retrieved_context_adequacy` | Whether the best possible context inside the retrieved candidate pool was good enough. | Separates retrieval-pool quality from final selection quality. |
| `answer_expected_alignment` | Whether the bot answer matches the expert expected answer. | Measures final answer usefulness. |
| `answer_groundedness` | Whether the bot answer stays grounded in the selected KB context. | Detects hallucinations and unsupported claims. |
| `language_compliance` | Whether the bot answer is in Slovak as required by the deployment. | Flags Czech or other-language drift as a production bug. |

Additional diagnostic fields from the judge:

| Field | Purpose |
| --- | --- |
| `issue_categories` | Multi-label failure tags for pattern aggregation. |
| `primary_root_cause` | Single dominant explanation for the case outcome. |
| `affected_teams` | Ownership mapping for backlog prioritization. |
| `optimal_enum_selection` | Whether the best ENUM set was available in the retrieved pool and selected correctly. |
| `missing_facts`, `hallucinated_claims` | Concrete evidence of content gaps vs unsupported answer content. |
| `candidate_pool_content_gap_identified`, `candidate_pool_content_gap_description` | Whether the retrieval pool itself lacked the needed KB content. |
| `retrieval_improvement_suggestion`, `reranker_improvement_suggestion`, `bot_improvement_suggestion`, `kb_improvement_suggestion`, `test_case_improvement_suggestion` | Team-specific next actions. |

## Executive Summary

## Actionable Insights

By this evaluation we want to gather answers to the following questions:

- Are failures mainly caused by answer generation and grounding, retriever candidate-pool quality, reranker selection quality, KB content gaps, or test-case labeling noise?
- When the final answer is bad, was the right information already present in the retrieved pool, or was it unavailable before reranking?
- Which rubric dimensions most often pull a case below the pass threshold, and are those drops isolated or compounded?
- How often do `case_scope` and `case_scope_fit` disagree, and do those disagreements point to ambiguous cases, routing problems, or dataset-labeling issues?
- Which issue categories and root causes concentrate by `case_scope`, `query_scope`, or affected team?
- What are the highest-leverage fixes for each team (`bot_agent`, `reranker`, `retrieval`, `kb_editors`, `test_set`) based on the judge suggestions and failure annotations?
- Which failing cases are true production defects versus acceptable non-KB or account-data-by-design cases that should not be counted as KB misses?

***

## Parameters

In [0]:
RUN_ON_DBX = False

In [0]:
from pathlib import Path
import yaml

YAML_FILE_NAME = "skkb_exp_001_baseline"
REASONING_EFFORT = "medium"
CHECKPOINT_SUFFIX = ""

if RUN_ON_DBX:
    repo_root = Path("/Workspace/Repos/Shared_HeyGeorge/ds-evals")
else:
    repo_root_candidates = [Path.cwd(), Path.cwd() / "ds-evals", *Path.cwd().parents]
    repo_root = next(
        (
            candidate
            for candidate in repo_root_candidates
            if (candidate / "configs" / "skkb" / f"{YAML_FILE_NAME}.yaml").exists()
        ),
        Path.cwd(),
    )

CONFIG_PATH = repo_root / "configs" / "skkb" / f"{YAML_FILE_NAME}.yaml"
CHECKPOINT_CSV = Path("checkpoints") / f"evals_{YAML_FILE_NAME}_{REASONING_EFFORT}{CHECKPOINT_SUFFIX}.csv"
REPORT_DIR = repo_root / "reports" / YAML_FILE_NAME
REPORT_PATH = REPORT_DIR / f"report{CHECKPOINT_SUFFIX}.html"

with CONFIG_PATH.open("r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

SCORE_COLS = [f"{dimension['id']}_score" for dimension in config["rubric"]["dimensions"]]

DIMENSION_WEIGHTS = {
    f"{dimension['id']}_score": float(dimension["weight"])
    for dimension in config["rubric"]["dimensions"]
}
print( f"--- SCORERS & WEIGHTS ---")
for dimension in config["rubric"]["dimensions"]:
    print(f"{dimension['id']}_score: {dimension['weight']}")

PASS_THRESHOLD = float(config["rubric"]["pass_threshold"])
TOTAL_WEIGHT = sum(DIMENSION_WEIGHTS.values())
MAX_WEIGHTED_SUM = 2 * TOTAL_WEIGHT
PASS_WEIGHTED_SUM = PASS_THRESHOLD * TOTAL_WEIGHT
LOSS_BUDGET = MAX_WEIGHTED_SUM - PASS_WEIGHTED_SUM


--- SCORERS & WEIGHTS ---
query_clarity_score: 1.0
case_scope_fit_score: 1.0
selection_semantic_relevance_score: 1.5
selected_context_sufficiency_score: 1.5
optimal_retrieved_context_adequacy_score: 1.5
answer_expected_alignment_score: 2.0
answer_groundedness_score: 2.0
language_compliance_score: 1.0


In [0]:
import ast
import json
import os
import sys
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
pd.set_option("display.width", None)
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import spearmanr

for module_dir in (Path.cwd(), Path.cwd() / "notebooks/skkb", Path.cwd() / "ds-evals/notebooks/skkb"):
    if (module_dir / "skkb_checkpoint.py").exists():
        sys.path.insert(0, str(module_dir))
        break
from skkb_checkpoint import read_checkpoint_csv

def _resolve_checkpoint_path(path: str | Path) -> Path:
    path = Path(path)
    if path.is_absolute():
        candidates = [path]
    else:
        candidates = [
            path,
            Path.cwd() / path,
            Path.cwd() / "notebooks/skkb" / path,
            Path.cwd() / "ds-evals/notebooks/skkb" / path,
        ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    searched = "\n".join(str(c) for c in candidates)
    raise FileNotFoundError(f"Could not find checkpoint CSV. Searched:\n{searched}")


checkpoint_path = _resolve_checkpoint_path(CHECKPOINT_CSV)
if not RUN_ON_DBX:
    repo_candidates = [p for p in checkpoint_path.parents if p.name == "ds-evals"]
    repo_root = repo_candidates[0] if repo_candidates else checkpoint_path.parent
    REPORT_DIR = str(repo_root / "reports" / YAML_FILE_NAME)
    REPORT_PATH = f"{REPORT_DIR}/report{CHECKPOINT_SUFFIX}.html"
df = read_checkpoint_csv(checkpoint_path)

print(f"checkpoint: {checkpoint_path}")
print(f"rows: {len(df)}   cols: {len(df.columns)}")
print("columns:", list(df.columns))


checkpoint: /Users/itacdonev/Developer/Erste/ds-evals/notebooks/skkb/checkpoints/evals_skkb_exp_001_baseline_medium.csv
rows: 484   cols: 84
columns: ['test_case_id', 'user_query', 'reranked_kb_context', 'reranked_enum_ids', 'retrieved_candidates_text', 'retrieved_enum_ids', 'expected_response', 'expected_enums', 'expected_enums_weights', 'bot_response', 'expert_score', 'expert_rationale', 'enum_relevance_score', 'execution_duration_ms', 'trace_id', 'request_time', 'reranker_system_prompt', 'reranker_user_prompt', 'reranker_model', 'reranker_token_usage', 'reranker_selected_ids', 'reranker_raw_selected_ids', 'reranker_valid_selected_ids', 'reranker_invalid_selected_ids', 'reranker_unselected_context_ids', 'reranker_selection_status', 'reranker_selection_violations', 'main_agent_prompt_hash', 'daily_banking_agent_prompt_hash', 'agents_called', 'tools_called', 'query_scope', 'reranker_selected_empty', 'trace_invariant_violations', 'kb_version', 'user_query_en', 'expected_response_en', 'b

Check for errors:

In [0]:
## Error check
if "error" in df.columns:
    err_mask = df["error"].astype(str).str.lower().isin({"true", "1", "1.0"})
    err_rows = df[err_mask]
    print(f"rows with error: {len(err_rows)}")
    if len(err_rows):
        cols = [c for c in ("test_case_id", "error", "error_message", "_csv_load_warning") if c in err_rows.columns]
        display(err_rows[cols].head(10))
else:
    print("no 'error' column")


rows with error: 1


,test_case_id,error,error_message,_csv_load_warning
151,Test case 152,True,"Expecting ',' delimiter: line 18 column 1 (cha...",short checkpoint row with 2 fields; treated as...


In [0]:
# TODO Check that optimal_enum_selection does not have hallucinations

## Processing of Results

To each test case we add the following derived attributes:

**1. Weighted sum - `weighted_sum` (float)**
This is the rubric-weighted total computed from the eight judge score columns:
$$
\begin{aligned}
\text{weighted sum} = &1.0 \cdot \text{query_clarity_score} + 1.0 \cdot \text{case_scope_fit_score} \\
&+ 1.5 \cdot \text{selection_semantic_relevance_score} + 1.5 \cdot \text{selected_context_sufficiency_score} \\
&+ 1.5 \cdot \text{optimal_retrieved_context_adequacy_score} + 2.0 \cdot \text{answer_expected_alignment_score} \\
&+ 2.0 \cdot \text{answer_groundedness_score} + 1.0 \cdot \text{language_compliance_score}
\end{aligned}
$$

**2. Weighted average score - `weighted_avg` (float)**
$$
\text{weighted average} = \frac{\text{weighted sum}}{11.5}
$$
The score range is [0, 2].

**3. Pass flag - `passed_eval` (boolean)**
A case passes if:

$$
\text{weighted average} \ge 1.4
$$
This is equivalent to:

$$
\text{weighted sum} \ge 1.4 \times 11.5 = 16.1
$$

With eight dimensions on a 0-2 scale, the perfect weighted sum is:
$$
2 \times 11.5 = 23.0
$$
So the pass threshold allows a loss budget of:
$$
23.0 - 16.1 = 6.9
$$

Interpretation of a one-level drop from 2 to 1:
- `answer_expected_alignment` or `answer_groundedness`: lose 2.0 weighted points
- `selection_semantic_relevance`, `selected_context_sufficiency`, or `optimal_retrieved_context_adequacy`: lose 1.5 weighted points
- `query_clarity`, `case_scope_fit`, or `language_compliance`: lose 1.0 weighted point

This makes the weighting intentionally stricter on final answer quality and grounding than on query quality or scope labeling. Rows with missing score values keep `weighted_sum`, `weighted_avg`, and `passed_eval` missing so parse errors are not silently counted as failures.

In [0]:
score_frame = df[SCORE_COLS].apply(pd.to_numeric, errors="coerce")
df[SCORE_COLS] = score_frame

weight_series = pd.Series(DIMENSION_WEIGHTS)
df["weighted_sum"] = score_frame.mul(weight_series, axis=1).sum(
    axis=1, min_count=len(SCORE_COLS)
)
df["weighted_avg"] = df["weighted_sum"] / TOTAL_WEIGHT
df["passed_eval"] = (df["weighted_avg"] >= PASS_THRESHOLD).astype("boolean")
df.loc[df["weighted_avg"].isna(), "passed_eval"] = pd.NA

scored_rows = df["weighted_avg"].notna()

print(f"Config path = {CONFIG_PATH}")
print(f"Pass threshold = {PASS_THRESHOLD}")
print(f"Total dimensions weight = {TOTAL_WEIGHT}")
print(f"Perfect weighted sum = {MAX_WEIGHTED_SUM:.1f}")
print(f"Required weighted sum for pass = {PASS_WEIGHTED_SUM:.1f}")
print(f"Loss budget from perfect = {LOSS_BUDGET:.1f}")
print(f"Rows with complete judge scores = {int(scored_rows.sum())}/{len(df)}")
print(
    f"Passing cases = {int(df.loc[scored_rows, 'passed_eval'].sum())}/{int(scored_rows.sum())} "
    f"({df.loc[scored_rows, 'passed_eval'].mean():.1%} of scored rows)"
)


Config path = /Users/itacdonev/Developer/Erste/ds-evals/configs/skkb/skkb_exp_001_baseline.yaml
Pass threshold = 1.4
Total dimensions weight = 11.5
Perfect weighted sum = 23.0
Required weighted sum for pass = 16.1
Loss budget from perfect = 6.9
Rows with complete judge scores = 483/484
Passing cases = 337/483 (69.8% of scored rows)


## Additional Analysis

In [0]:
df.head(1)

,test_case_id,user_query,reranked_kb_context,reranked_enum_ids,retrieved_candidates_text,retrieved_enum_ids,expected_response,expected_enums,expected_enums_weights,bot_response,...,answer_expected_alignment_score,answer_expected_alignment_reasoning,answer_groundedness_score,answer_groundedness_reasoning,language_compliance_score,language_compliance_reasoning,_csv_record_number,_csv_load_warning,error,error_message
0,Test case 1,"Dá sa na sporenie vložiť aj hotovosť, alebo le...",[KB_GAI_SK_SK_2026-03-17_17h26]\nSAVING@ACCOUN...,"[""SAVING@ACCOUNT_MOVEMENTS"", ""SAVING@KIDS_WITH...",SAVING@ACCOUNT_MOVEMENTS: **Vklady na sporiaci...,"[""SAVING@ACCOUNT_MOVEMENTS"", ""SAVING@KIDS_DISP...","Marek, peniaze na sporenie môžete vložiť klasi...","[""SAVING@ACCOUNT_MOVEMENTS""]","{""SAVING@ACCOUNT_MOVEMENTS"": 4}",Na „bežné“ sporenie si môžete vložiť aj hotovo...,...,2,"The bot states both cash and transfer, mention...",2,All claims are supported by the selected ENUMs...,2,The response is fully in Slovak.,2,,False,


### Comparison of case_scope_fit and case_scope

Keep this section, but treat it as a validation check rather than a major analysis block. Raw counts are useful, but a row-normalized view and a small review table for `case_scope_fit_score < 2` make the disagreements much easier to inspect.

In [0]:
case_scope_label = df["case_scope"].fillna("<missing>").replace("", "<blank>")
scope_counts = pd.crosstab(df["case_scope_fit_score"], case_scope_label, dropna=False)
scope_share = pd.crosstab(
    df["case_scope_fit_score"], case_scope_label, normalize="index", dropna=False
).round(3)

display(scope_counts)
display(scope_share)

scope_review_cols = [
    "test_case_id",
    "user_query",
    "case_scope",
    "query_scope",
    "case_scope_fit_score",
    "case_scope_fit_reasoning",
]
# display(
#     df.loc[df["case_scope_fit_score"] < 2, scope_review_cols]
#     .sort_values(["case_scope", "test_case_id"])
#     .reset_index(drop=True)
# )


case_scope,<blank>,account_data_by_design,ambiguous,kb_answerable
case_scope_fit_score,,,,
1.0,0,1,13,8
2.0,0,3,0,458
NaN,1,0,0,0


case_scope,<blank>,account_data_by_design,ambiguous,kb_answerable
case_scope_fit_score,,,,
1.0,0.0,0.045,0.591,0.364
2.0,0.0,0.007,0.000,0.993
NaN,1.0,0.000,0.000,0.000


### Queries sent to the KB search tool

In [0]:
def extract_tool_info(cell):
    if pd.isna(cell):
        return pd.Series([None, [], 0])

    try:
        data = json.loads(cell) if isinstance(cell, str) else cell

        if isinstance(data, list) and len(data) > 0:
            first = data[0]
            tool_name = first.get("name")
            queries = first.get("inputs", {}).get("queries", [])
            return pd.Series([tool_name, queries, len(queries)])

    except Exception:
        pass

    return pd.Series([None, [], 0])

df[["tool_name", "kb_search_queries", "kb_search_queries_size"]] = (
    df["tools_called"].apply(extract_tool_info)
)

In [0]:
# Enter a test case ID
tid = 1
df[df["test_case_id"] == f"Test case {tid}"][["tool_name", "kb_search_queries_size"]]

,tool_name,kb_search_queries_size
0,knowledge_search,5


In [0]:
df["kb_search_queries_size"].describe()

count    484.000000
mean       4.334711
std        1.798191
min        0.000000
25%        5.000000
50%        5.000000
75%        5.000000
max        6.000000
Name: kb_search_queries_size, dtype: float64